# 03 微调策略比较（E2）

**目标：** 固定已选骨干，对比冻结骨干、部分微调和整体微调。

**前置条件：** E1 已完成；在 `configs/experiments.yaml` 中填写 `selected_backbone`；固定类别和数据清单保持一致。

**输入：** 基础配置、选定骨干及 `finetuning_strategies`。

**输出：** `outputs/runs/e2_*/` 的实际实验记录及最佳权重；依据验证集确定 `selected_finetuning`。

默认关闭训练。每种策略都从相同来源的预训练权重开始，不接着其他策略的权重继续训练。

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

def summarize_validation(plan_list):
    records = []
    for plan in plan_list:
        run_dir = PROJECT_ROOT / "outputs" / "runs" / plan["run_id"]
        history = read_csv(run_dir / "history.csv")
        if not history:
            continue
        config_path = run_dir / "config.yaml"
        actual_config = load_config(config_path) if config_path.exists() else plan["config"]
        monitor = actual_config["training"].get("monitor", "macro_f1")
        key = "val_" + monitor
        best = min(history, key=lambda row: float(row[key])) if monitor == "loss" else max(
            history, key=lambda row: float(row[key]))
        records.append({"run_id": plan["run_id"], "recorded_epochs": len(history),
                        "monitor": monitor, "best_epoch": int(best["epoch"]),
                        "val_loss": float(best["val_loss"]),
                        "val_accuracy": float(best["val_accuracy"]),
                        "val_macro_f1": float(best["val_macro_f1"]),
                        "total_training_seconds": float(history[-1]["elapsed_seconds"])})
    return records


## 1. 固定骨干，生成策略计划

此轮保持基础学习率和预算一致，记录各策略表现。若之后需要优化学习率，应将其记录为独立的超参数实验，并使用相同验证规则。

In [ ]:
RUN_TRAINING = False
selected_backbone = EXPERIMENTS.get("selected_backbone")
plans = []
if not selected_backbone:
    print("请先依据 E1 验证结果，在 experiments.yaml 中填写 selected_backbone。")
else:
    for strategy in EXPERIMENTS.get("finetuning_strategies", ["freeze", "partial", "full"]):
        for seed in EXPERIMENTS.get("seeds", [BASE.get("seed", 42)]):
            config = deepcopy(BASE)
            config["model"]["name"] = selected_backbone
            config["model"]["finetuning"] = strategy
            config["seed"] = int(seed)
            plans.append({"run_id": f"e2_{selected_backbone}_{strategy}_seed{seed}", "config": config})
show_table([{"run_id": p["run_id"], "model": p["config"]["model"]["name"],
             "finetuning": p["config"]["model"]["finetuning"],
             "lr": p["config"]["training"]["lr"], "epochs": p["config"]["training"]["epochs"]}
            for p in plans])

## 2. 运行策略比较

部分微调时具体解冻层由 `src/models.py` 中对应骨干的规则定义；论文中应记录解冻范围。模型参数量与可训练参数量会随实验保存。

In [ ]:
if not plans:
    print("尚无可运行计划：先选定骨干。")
elif not RUN_TRAINING:
    print("当前只展示计划。确认后将 RUN_TRAINING 设为 True。")
elif not LABELS.get("classes"):
    print("请先完成共同类别及固定划分。")
else:
    from src.train import run_experiment
    for plan in plans:
        run_dir = run_experiment(PROJECT_ROOT, plan["config"], plan["run_id"])
        print("已保存：", run_dir)

In [ ]:
records = summarize_validation(plans)
show_table(records)
print("当前选定微调策略：", EXPERIMENTS.get("selected_finetuning"))


## 结论记录

- 冻结、部分微调、整体微调的验证表现及波动：待填写。
- 各策略的训练时间、可训练参数及过拟合现象：待填写。
- 选定策略和依据：待填写，并更新 `selected_finetuning`。
- 后续是否需要独立学习率实验：待填写。